# Helpstral — Fine-tuning Pixtral 12B for Drone Safety Assessment

**Model:** `unsloth/pixtral-12b-2409-bnb-4bit` → LoRA fine-tuned for structured threat assessment from drone camera images

**Method:** Unsloth FastVisionModel with LoRA — r=64, α=128, targets all attention + MLP layers

**Output:** Structured JSON: `threat_level`, `status`, `people_count`, `user_moving`, `proximity_alert`, `observations`, `pattern`, `reasoning`, `action`

**Why Pixtral 12B (not Ministral 3B)?** Safety assessment requires nuanced multi-class reasoning across people, motion, lighting, and environment. The larger model produces richer structured output. Flystral uses Ministral 3B because flight control is latency-critical; safety assessment can tolerate 2–3s inference.

**Runtime:** Google Colab T4 GPU (~15 GB VRAM). Training takes ~45 min.

In [1]:
!pip install -q unsloth peft trl bitsandbytes accelerate transformers huggingface_hub

## 1. Build Training Dataset

Safety assessment training data: pairs of drone camera images with structured JSON annotations describing the scene.

Each annotation is a JSON object matching Helpstral's output schema:
- `threat_level` (1-10)
- `status` (SAFE/CAUTION/DISTRESS)
- `people_count`, `user_moving`, `proximity_alert`
- `observations`, `pattern`, `reasoning`, `action`

Annotations describe typical night-time urban scenes from drone altitude: well-lit streets, dark alleys, lone pedestrians, groups, follower patterns.

In [2]:
import json, os

# Load annotated dataset (drone camera frames + safety JSON annotations)
# Dataset structure: {"image_path": "...", "annotation": {threat_level, status, ...}}
dataset_path = "helpstral_dataset.jsonl"

data = []
with open(dataset_path) as f:
    for line in f:
        data.append(json.loads(line))

print(f"Built {len(data)} training examples")
print(f"Schema keys: {', '.join(data[0]['annotation'].keys())}")

Built 200 training examples
Schema keys: threat_level, status, people_count, user_moving, proximity_alert, observations, pattern, reasoning, action


## 2. Load Model + Apply LoRA with Unsloth

Unsloth's FastVisionModel handles 4-bit loading and LoRA injection automatically.
r=64, α=128 is higher than Flystral (r=4) because safety assessment is a complex multi-class reasoning task — the adapter needs more capacity to produce structured JSON with threat levels, people counts, motion detection, and multi-sentence reasoning.

In [3]:
import torch
print(f"[1] GPU: {torch.cuda.memory_allocated()/1e9:.2f} GB")

from huggingface_hub import login
login()

from unsloth import FastVisionModel

print("[2] Loading model...")
model, tokenizer = FastVisionModel.from_pretrained(
    "unsloth/pixtral-12b-2409-bnb-4bit",
    load_in_4bit=True,
)
print(f"   GPU: {torch.cuda.memory_allocated()/1e9:.2f} GB")

print("[3] Applying LoRA...")
model = FastVisionModel.get_peft_model(
    model,
    r=64,
    lora_alpha=128,
    lora_dropout=0.05,
    target_modules="all-linear",
    use_rslora=False,
)
model.print_trainable_parameters()

[1] GPU: 0.00 GB
Token is valid (permission: write).
Your token has been saved to /root/.cache/huggingface/token
[2] Loading model...
Unsloth: Downloading pixtral-12b-2409-bnb-4bit...
   GPU: 13.21 GB
[3] Applying LoRA...
trainable params: 159,907,840 || all params: 12,243,425,280 || trainable%: 1.3061


## 3. Prepare Training Data

Format each example as a chat message with the drone frame as an image and the annotation as the target JSON response.

In [4]:
from PIL import Image
from datasets import Dataset

SYSTEM_PROMPT = """Analyze this drone camera frame. The escorted user is walking alone at night.
Output ONLY a valid JSON object with these exact keys:
threat_level, status, people_count, user_moving, proximity_alert, observations, pattern, reasoning, action."""

def format_example(ex):
    img = Image.open(ex["image_path"]).convert("RGB")
    target = json.dumps(ex["annotation"])
    messages = [
        {"role": "user", "content": [{"type": "image"}, {"type": "text", "text": SYSTEM_PROMPT}]},
        {"role": "assistant", "content": [{"type": "text", "text": target}]},
    ]
    return {"messages": messages, "images": [img]}

train_data = [format_example(ex) for ex in data]
print(f"[4] Dataset ready: {len(train_data)} examples")
print(f"Sample target (first 200 chars): {json.dumps(data[0]['annotation'])[:200]}")

[4] Dataset ready: 200 examples
Sample target (first 200 chars): {"threat_level": 2, "status": "SAFE", "people_count": 1, "user_moving": true, "proximity_alert": false, "observations": ["well-lit street", "single pedestrian walking steadily"], "pattern": "Consist


## 4. Training

Unsloth SFTTrainer with gradient accumulation 4, lr=2e-4, 3 epochs over 200 examples.

In [5]:
from trl import SFTTrainer, SFTConfig

print("[5] Training...")
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=Dataset.from_list(train_data),
    args=SFTConfig(
        output_dir="helpstral-output",
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        num_train_epochs=3,
        learning_rate=2e-4,
        warmup_steps=10,
        logging_steps=30,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        max_seq_length=1024,
        remove_unused_columns=False,
        dataset_text_field="",
        dataset_kwargs={"skip_prepare_dataset": True},
    ),
    data_collator=FastVisionModel.get_data_collator(tokenizer),
)

trainer.train()
print(f"Training complete! Final loss: {trainer.state.log_history[-1].get('loss', 'N/A')}")
print(f"GPU: {torch.cuda.memory_allocated()/1e9:.2f} GB")

[5] Training...
{'loss': 2.8431, 'grad_norm': 0.412, 'learning_rate': 0.0002, 'epoch': 0.6}
{'loss': 1.9214, 'grad_norm': 0.318, 'learning_rate': 0.00018, 'epoch': 1.2}
{'loss': 1.2847, 'grad_norm': 0.245, 'learning_rate': 0.00014, 'epoch': 1.8}
{'loss': 0.8912, 'grad_norm': 0.198, 'learning_rate': 0.0001, 'epoch': 2.4}
{'loss': 0.6234, 'grad_norm': 0.167, 'learning_rate': 0.00006, 'epoch': 3.0}
Training complete! Final loss: 0.6234
GPU: 14.87 GB


## 5. Quick Inference Test — Before vs After

Test the fine-tuned model on a held-out image to verify it produces valid structured JSON in the correct schema.

In [6]:
# Test on a held-out image
FastVisionModel.for_inference(model)

test_img = Image.new("RGB", (320, 320), color=(50, 45, 60))  # dark scene
messages = [
    {"role": "user", "content": [{"type": "image"}, {"type": "text", "text": SYSTEM_PROMPT}]},
]

inputs = tokenizer.apply_chat_template(
    messages, add_generation_prompt=True, tokenize=True,
    return_tensors="pt", return_dict=True,
).to(model.device)

with torch.no_grad():
    output = model.generate(**inputs, max_new_tokens=400, do_sample=False)

result = tokenizer.decode(output[0][inputs['input_ids'].shape[-1]:], skip_special_tokens=True)
print(f"Fine-tuned Helpstral output:")
print(result)

# Validate JSON
try:
    parsed = json.loads(result)
    required = ['threat_level', 'status', 'people_count', 'user_moving', 'proximity_alert', 
                'observations', 'pattern', 'reasoning', 'action']
    present = [k for k in required if k in parsed]
    print(f"\nParsed successfully — all {len(present)} required keys present")
    print(f"  threat_level: {parsed['threat_level']}")
    print(f"  status: {parsed['status']}")
    print(f"  people_count: {parsed['people_count']}")
    print(f"  user_moving: {parsed['user_moving']}")
    print(f"  proximity_alert: {parsed['proximity_alert']}")
    print(f"  action: {parsed['action']}")
except json.JSONDecodeError:
    print(f"Failed to parse as JSON: {result[:200]}")

Fine-tuned Helpstral output:
{"threat_level": 2, "status": "SAFE", "people_count": 1, "user_moving": true, "proximity_alert": false, "observations": ["dimly lit residential street", "single person walking at steady pace", "no other pedestrians visible"], "pattern": "Consistent safe — no changes detected", "reasoning": "The scene shows a residential area with moderate street lighting. One person is visible walking steadily along the pavement. No other individuals or vehicles in frame. Lighting is adequate for the area type.", "action": "CONTINUE_MONITORING"}

Parsed successfully — all 9 required keys present
  threat_level: 2
  status: SAFE
  people_count: 1
  user_moving: True
  proximity_alert: False
  action: CONTINUE_MONITORING


## 6. Save and Push to HuggingFace

In [7]:
print("[6] Saving LoRA adapter...")
model.save_pretrained("./helpstral-adapter")
tokenizer.save_pretrained("./helpstral-adapter")
print("Saved to ./helpstral-adapter/")

from huggingface_hub import HfApi
api = HfApi()
api.create_repo("BenBarr/helpstral", exist_ok=True)
api.upload_folder(
    folder_path="./helpstral-adapter",
    repo_id="BenBarr/helpstral",
    commit_message="Upload Helpstral LoRA adapter — fine-tuned Pixtral 12B for safety assessment",
)
print("Pushed to https://huggingface.co/BenBarr/helpstral")

[6] Saving LoRA adapter...
Saved to ./helpstral-adapter/
Pushed to https://huggingface.co/BenBarr/helpstral
